# Dataplex LLM System Prompt Generator

This notebook generates **Instructions/Context for BigQuery Conversational Analytics & VertexAI Agents**
based on metadata JSON files stored in GCS.

It produces a complete system prompt for Text-to-SQL LLM agents that includes:
1. **Schema structure** - What columns exist and their purposes
2. **Business semantics** - What business concepts map to which columns
3. **Usage rules** - Which fields to prefer/avoid for specific query types
4. **Calculation logic** - How derived metrics are computed

## 1. Install Required Packages

In [ ]:
%pip install --quiet google-cloud-storage
%pip install --quiet google-cloud-dataplex

## 2. Authentication (for Google Colab)

In [ ]:
# Uncomment if running in Google Colab
from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated successfully!")

## 3. Configuration

In [ ]:
# === CONFIGURATION ===
# Update these values for your environment

PROJECT_ID = "ordertake-datalake-gcp-pilot"
LOCATION = "us"  # or "global" depending on your glossary location
GLOSSARY_ID = "ot-glossary-v6"

BUCKET_NAME = "ordertake-datalake-gcp-pilot-landing"
FILE_NAME = "4_business_glossary_and_KPI_Dataplex_vs_BigQuery.json"

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Glossary ID: {GLOSSARY_ID}")
print(f"Bucket: {BUCKET_NAME}")
print(f"File: {FILE_NAME}")

## 4. Initialize Clients

In [ ]:
import json
from google.cloud import storage

# Initialize Storage Client (needed to load metadata JSON from GCS)
storage_client = storage.Client(project=PROJECT_ID)

print("✅ Storage Client initialized.")

## 5. Context Generator for LLM Prompts

In [ ]:
def generate_llm_system_prompt(
    metadata: dict,
    agent_name: str = "SQL Assistant",
    domain: str = "Automotive Order Take"
) -> str:
    """
    Generates a complete system prompt for Text-to-SQL LLM agents.

    This can be used as the system prompt for:
    - VertexAI Agents
    - Custom LangChain agents
    - OpenAI function calling
    - Any Text-to-SQL implementation

    Args:
        metadata: The metadata dictionary
        agent_name: Name of the agent
        domain: Business domain description

    Returns:
        Complete system prompt string
    """
    config = metadata.get("config", {})
    tech_meta = metadata.get("bigquery_technical_metadata", {})
    glossary = metadata.get("dataplex_business_glossary", {})

    dataset = config.get("dataset", "unknown_dataset")
    table = config.get("table", "unknown_table")
    table_desc = tech_meta.get("description", "")
    columns = tech_meta.get("columns", {})
    terms = glossary.get("terms", [])

    # Build column schema string
    schema_lines = []
    for col_name, col_meta in columns.items():
        desc = col_meta.get("description", "")
        dtype = col_meta.get("data_type_hint", "Unknown")
        schema_lines.append(f"  - {col_name} ({dtype}): {desc}")
    schema_str = "\n".join(schema_lines)

    # Build business term mappings
    term_lines = []
    for term in terms:
        name = term.get("business_term", "")
        definition = term.get("definition", "")
        linked = term.get("linked_assets", [])
        col = linked[0].split(".")[-1] if linked else "N/A"
        term_lines.append(f'  - "{name}" → Column: {col} | {definition}')
    terms_str = "\n".join(term_lines)

    prompt = f"""You are {agent_name}, a SQL Assistant specialized in {domain} datasets.

YOUR TASK:
Convert natural language questions into valid BigQuery SQL queries.

DATABASE SCHEMA:
Table: `{dataset}.{table}`
Description: {table_desc}

Columns:
{schema_str}

BUSINESS TERM DICTIONARY:
When users mention these business terms, map them to the corresponding columns:
{terms_str}

IMPORTANT RULES:
1. Columns starting with 'DIM_' are dimensions; 'M_' are metrics.
2. **CRITICAL:** Always treat DIM_YEARMONTH as a String. Never format it with thousands separators (commas).
3. **OUTPUT LIMITATION:** Do not display the raw technical query result or a "Data Analysis" preview table. Only provide the final formatted "Tabular View" described in the response format below.
4. Pre-calculated KPI columns must be used directly instead of recalculating.
5. Always qualify the table with the full path: `{dataset}.{table}`


RESPONSE FORMAT:
When asked a question: Provide the valid BigQuery SQL query.
"""

    return prompt

## 6. Generate the System Prompt

In [ ]:
# Load metadata from GCS
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(FILE_NAME)
json_content = blob.download_as_text()
gcs_metadata = json.loads(json_content)

# Generate the system prompt
print("\n" + "="*60)
print("🤖 LLM SYSTEM PROMPT FOR TEXT-TO-SQL AGENT")
print("="*60 + "\n")

system_prompt = generate_llm_system_prompt(
    gcs_metadata,
    agent_name="Ordertake Agent",
    domain="Automotive Order Take"
)
print(system_prompt)